In [1]:
# Import libraries and StackSats classes needed for exporting strategy weights, merging data, and plotting results.
import sys
import polars as pl
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.baselines.uniform import UniformStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

In [2]:
# Initialize the runner and load the prepared Bitcoin analytics parquet manually.
runner = StrategyRunner()

btc_df = pl.read_parquet(STACKSATS_DATA_PATH).with_columns(pl.col("date").cast(pl.Datetime))

print("Loaded rows:", btc_df.height)
print(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Loaded rows: 5689
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
btc_full = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_full.height)
print(
    btc_full.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [4]:
cycle_results = {}

for cycle in plots.calendar_cycles:
    result = strategy_utils.process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_full,
        runner=runner,
        dynamic_strategy=MomentumStrategy(),
        total_budget_usd=1000.0,
        top_buy_quantile=0.90
    )

    cycle_results[cycle["label"]] = result


Processing Cycle 1: 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows
2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows

Processing Cycle 2: 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└────────

In [5]:
cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Combine all 4 cycles into one DataFrame
combined_plot_df = pd.concat(
    [r["plot_df"] for r in cycle_results.values()]
).sort_values("date").reset_index(drop=True)

# per cycle plots
cycle_plots = plots.plot_strategy_by_cycle(combined_plot_df, cols, "Momentum")
plt.show()

In [6]:
# Build monthly table for each cycle.
def build_monthly_table(result: dict):
    merged = result["merged"]

    monthly = (
        merged
        .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
        .group_by("month")
        .agg([
            pl.col("price_usd").mean().alias("avg_price_usd"),
            pl.col("dynamic_weight_raw").sum().alias("total_dynamic_weight"),
            pl.col("baseline_weight_raw").sum().alias("total_baseline_weight"),
            pl.col("sats_accum_dynamic").sum().alias("sats_accum_dynamic"),
            pl.col("sats_accum_baseline").sum().alias("sats_accum_baseline"),
            pl.col("dynamic_usd").sum().alias("total_dynamic_usd"),
            pl.col("baseline_usd").sum().alias("total_baseline_usd"),
        ])
        .with_columns([
            (pl.col("sats_accum_dynamic") / pl.col("total_dynamic_usd")).alias("avg_sats_per_dollar_dynamic"),
            (pl.col("sats_accum_baseline") / pl.col("total_baseline_usd")).alias("avg_sats_per_dollar_baseline"),
        ])
        .sort("month")
    )

    monthly_pd = monthly.to_pandas()
    monthly_pd["month"] = monthly_pd["month"].dt.strftime("%Y-%m")
    monthly_pd["avg_price_usd"] = monthly_pd["avg_price_usd"].round(2)

    for col in [
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]:
        monthly_pd[col] = monthly_pd[col].round(4)

    monthly_pd = monthly_pd[[
        "month",
        "avg_price_usd",
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]]

    return monthly_pd

In [7]:
# Create monthly tables for all 4 cycles.
cycle_monthly_tables = {
    label: build_monthly_table(result)
    for label, result in cycle_results.items()
}

In [8]:
# per year plots
year_plots = plots.plot_strategy_by_year(combined_plot_df, cols, "Momentum")
plt.show()

In [9]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913
1,2013-05-09,111.97,0.002412,0.000913
2,2011-07-09,14.39,0.002234,0.000913
3,2013-05-08,113.47,0.002166,0.000913
4,2011-07-08,14.35,0.002163,0.000913
5,2013-05-03,94.26,0.002053,0.000913
6,2013-05-07,111.43,0.002044,0.000913
7,2011-07-10,15.08,0.001960,0.000913
8,2011-08-06,7.82,0.001959,0.000913
9,2011-08-07,7.72,0.001906,0.000913


In [10]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2011-07-09,14.39,0.002234,0.000913
1,2011-07-08,14.35,0.002163,0.000913
2,2011-07-10,15.08,0.001960,0.000913
3,2011-08-06,7.82,0.001959,0.000913
4,2011-08-07,7.72,0.001906,0.000913
5,2011-08-08,7.74,0.001897,0.000913
6,2011-09-17,4.77,0.001873,0.000913
7,2011-09-16,4.81,0.001872,0.000913
8,2011-09-15,4.97,0.001866,0.000913
9,2011-09-18,5.11,0.001846,0.000913


In [11]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2011, 12, 31))
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2011-12-31,4.58,0.000003,0.000913


In [12]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913
1,2012-02-19,4.38,0.001239,0.000913
2,2012-02-15,4.69,0.001234,0.000913
3,2012-02-17,4.67,0.001222,0.000913
4,2012-02-14,4.89,0.001215,0.000913
5,2012-02-20,4.44,0.001200,0.000913
6,2012-02-18,4.23,0.001188,0.000913
7,2012-10-26,9.88,0.001179,0.000913
8,2012-02-21,4.58,0.001171,0.000913
9,2012-11-02,10.52,0.001166,0.000913


In [13]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2012, 12, 31))
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913


In [14]:
cycle1_2012 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("date")
    .with_columns([
        pl.col("dynamic_weight").cum_sum().alias("cum_dynamic_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2012.tail(10).to_pandas()

,date,price_usd,dynamic_weight_raw,baseline_weight_raw,dynamic_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_dynamic_weight,cum_baseline_weight
0,2012-12-22,13.20,0.002609,0.00274,0.000870,0.000913,0.869533,0.913242,0.065874,0.069185,6.587370e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.305998,0.325114
1,2012-12-23,13.14,0.002682,0.00274,0.000894,0.000913,0.894139,0.913242,0.068047,0.069501,6.804711e+06,6.950091e+06,7.610350e+06,7.610350e+06,0.306892,0.326027
2,2012-12-24,13.23,0.002691,0.00274,0.000897,0.000913,0.897107,0.913242,0.067809,0.069028,6.780856e+06,6.902812e+06,7.558579e+06,7.558579e+06,0.307789,0.326941
3,2012-12-25,13.24,0.002703,0.00274,0.000901,0.000913,0.901143,0.913242,0.068062,0.068976,6.806217e+06,6.897598e+06,7.552870e+06,7.552870e+06,0.308690,0.327854
4,2012-12-26,13.18,0.002691,0.00274,0.000897,0.000913,0.896891,0.913242,0.068049,0.069290,6.804940e+06,6.928999e+06,7.587253e+06,7.587253e+06,0.309587,0.328767
5,2012-12-27,13.20,0.002655,0.00274,0.000885,0.000913,0.885124,0.913242,0.067055,0.069185,6.705482e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.310472,0.329680
6,2012-12-28,13.18,0.002696,0.00274,0.000899,0.000913,0.898595,0.913242,0.068179,0.069290,6.817869e+06,6.928999e+06,7.587253e+06,7.587253e+06,0.311371,0.330594
7,2012-12-29,13.11,0.002747,0.00274,0.000916,0.000913,0.915585,0.913242,0.069839,0.069660,6.983866e+06,6.965995e+06,7.627765e+06,7.627765e+06,0.312286,0.331507
8,2012-12-30,13.20,0.002741,0.00274,0.000914,0.000913,0.913640,0.913242,0.069215,0.069185,6.921518e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.313200,0.332420
9,2012-12-31,13.24,0.060400,0.00274,0.020133,0.000913,20.133280,0.913242,1.520640,0.068976,1.520640e+08,6.897598e+06,7.552870e+06,7.552870e+06,0.333333,0.333333


In [15]:
(
    cycle1_2012
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("dynamic_weight").sum().alias("dynamic_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,dynamic_sum_before_last_day,baseline_sum_before_last_day
0,0.3132,0.33242


In [16]:
cycle1_2011 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("date")
    .with_columns([
        pl.col("dynamic_weight").cum_sum().alias("cum_dynamic_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2011.tail(10).to_pandas()

,date,price_usd,dynamic_weight_raw,baseline_weight_raw,dynamic_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_dynamic_weight,cum_baseline_weight
0,2011-12-22,3.79,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000880,0.240961,87950.747587,2.409610e+07,2.638522e+07,2.638522e+07,0.333303,0.325114
1,2011-12-23,3.90,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000855,0.234165,85470.085473,2.341646e+07,2.564103e+07,2.564103e+07,0.333307,0.326027
2,2011-12-24,3.92,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000850,0.232970,85034.013608,2.329699e+07,2.551020e+07,2.551020e+07,0.333310,0.326941
3,2011-12-25,4.14,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000805,0.220590,80515.297928,2.205899e+07,2.415459e+07,2.415459e+07,0.333313,0.327854
4,2011-12-26,4.04,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000825,0.226050,82508.250850,2.260500e+07,2.475248e+07,2.475248e+07,0.333317,0.328767
5,2011-12-27,4.02,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000829,0.227175,82918.739661,2.271746e+07,2.487562e+07,2.487562e+07,0.333320,0.329680
6,2011-12-28,4.14,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000805,0.220590,80515.297928,2.205899e+07,2.415459e+07,2.415459e+07,0.333323,0.330594
7,2011-12-29,4.22,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000790,0.216408,78988.941566,2.164081e+07,2.369668e+07,2.369668e+07,0.333327,0.331507
8,2011-12-30,4.19,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000796,0.217958,79554.494848,2.179575e+07,2.386635e+07,2.386635e+07,0.333330,0.332420
9,2011-12-31,4.58,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000728,0.199398,72780.203793,1.993978e+07,2.183406e+07,2.183406e+07,0.333333,0.333333


In [17]:
(
    cycle1_2011
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("dynamic_weight").sum().alias("dynamic_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,dynamic_sum_before_last_day,baseline_sum_before_last_day
0,0.333333,0.333333
